# Logistic Regression — Invented from Scratch
> *"Don't memorize. Understand why. Then the formula writes itself."*

---

## 1. Sigmoid Function

### The Problem
Linear regression outputs ANY number — 3000, -500, anything. Meaningless for YES/NO questions.

We don't want YES or NO directly. We want — **how CERTAIN are we of YES?**
That certainty must live between 0 and 1. Always. That's probability.

### Why Not a Threshold?
Two problems:
- **Kills confidence** — model outputting 0.51 and 0.99 both become "YES". But 0.99 is WAY more certain. Threshold throws that away. Dangerous for a doctor.
- **Outliers break everything** — one patient with extreme tumor size tilts the line, misclassifies everyone else.

### What We Actually Need
> A function that takes ANY number from −∞ to +∞ and smoothly squishes it into 0 to 1. Keeping confidence. Ignoring outlier drama.

### How We Built It
$e^x$ is always positive. Grows to ∞ on right, shrinks to 0 on left. Good raw material.

Take inverse idea → $\frac{1}{e^x}$ — goes to 0 for large x. Close but opposite of what we want.

Small twist — use $e^{-x}$ and add 1 to bottom:

$$\frac{1}{1 + e^{-x}}$$

**Check:**

| x | $e^{-x}$ | $1 + e^{-x}$ | Output |
|---|----------|--------------|--------|
| +∞ | 0 | 1 | **1** ✅ |
| 0 | 1 | 2 | **0.5** ✅ |
| −∞ | ∞ | ∞ | **0** ✅ |

### Formula
$$\boxed{\sigma(z) = \frac{1}{1 + e^{-z}}}$$

### One Line
> **Sigmoid doesn't make decisions. It measures confidence. The doctor makes the decision.**

---
## 2. Logistic Regression Function

### Intuition
Linear regression gives raw signal. Sigmoid converts that signal into probability.

So just plug linear regression INTO sigmoid:

**Step 1 — Linear part:**
$$z = \vec{w} \cdot \vec{x} + b$$

**Step 2 — Sigmoid squishes it:**
$$f(x) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

### Formula
$$\boxed{f_{\vec{w},b}(x) = \frac{1}{1 + e^{-(\vec{w} \cdot \vec{x} + b)}} = P(\text{yes} | x)}$$

### The Doctor Story
```
Tumor size (raw number)
    ↓  linear part
z = wx + b (any number)
    ↓  sigmoid
P(cancer | tumor size) = 0.91 (probability)
    ↓  threshold at 0.5
Doctor's decision → YES / NO
```

### One Line
> **Logistic Regression = Linear Regression wearing a Sigmoid mask.**

---
## 3. Maximum Likelihood Estimation (MLE)

### Intuition
You flip a coin 5 times. Get H, H, T, H, T.
You don't know if coin is fair. What value of $p$ (probability of Head) makes THIS sequence most likely?

Each flip is independent → multiply probabilities:
$$L = p \times p \times (1-p) \times p \times (1-p) = p^3(1-p)^2$$

What $p$ maximizes this? **$p = 3/5 = 0.6$** — exactly what you observed!

### Applied to Cancer
Instead of coins → patients. Instead of fixed $p$ → sigmoid gives different $p$ per patient.

> **MLE asks: What values of w and b make sigmoid output the highest probability for ALL correct diagnoses?**

### The Likelihood Function
For $m$ patients — multiply all their probabilities:
$$L(w,b) = \prod_{i=1}^{m} p^{(i)^{y^{(i)}}} (1-p^{(i)})^{1-y^{(i)}}$$

- Patient HAS cancer ($y=1$) → contributes $p^{(i)}$
- Patient has NO cancer ($y=0$) → contributes $(1-p^{(i)})$

### One Line
> **MLE = find the parameters that make what actually happened as likely as possible.**

---
## 4. Why Log? — The Log Trick

### The Problem with Raw MLE
Multiplying 1000 probabilities (all between 0 and 1) together:
$$0.9 \times 0.7 \times 0.8 \times \dots \text{(1000 times)} = 0.000000000001$$
Number vanishes. Computer stores it as 0. **Numerical underflow.** 💀

### The Fix — Logarithm
Log turns multiplication into addition:
$$\log(A \times B) = \log(A) + \log(B)$$

**Proof** (from first principles):
- Let $A = e^a$ → $\ln(A) = a$
- Let $B = e^b$ → $\ln(B) = b$
- $A \times B = e^a \times e^b = e^{a+b}$
- $\ln(A \times B) = a + b = \ln(A) + \ln(B)$ ✅

Now instead of multiplying 1000 tiny numbers → **add 1000 normal numbers.** Computer happy. 😄

### Log of probabilities is always negative
| $p$ | $\log(p)$ |
|-----|----------|
| 0.99 | -0.01 |
| 0.5 | -0.69 |
| 0.01 | -4.6 |

Adding them → big negative number. We want to MAXIMIZE likelihood but gradient descent MINIMIZES.

**Flip the sign** → Negative Log Likelihood → now we MINIMIZE it. ✅

---
## 5. Loss Function

### Why Not MSE (Squared Error)?
MSE worked for linear regression because cost bowl was smooth and convex.

But sigmoid is non-linear. Plugging sigmoid into MSE creates a **bumpy, non-convex bowl** with many local minimums. Gradient descent gets stuck. 💀

### What We Need
A loss that:
- Model correct AND confident → tiny loss
- Model wrong AND confident → HUGE punishment

### Deriving the Loss
When patient HAS cancer ($y = 1$):
- $p = 0.99$ → $-\log(0.99) = 0.01$ tiny ✅
- $p = 0.01$ → $-\log(0.01) = 4.6$ huge 💀

When patient has NO cancer ($y = 0$):
- $p = 0.01$ → $-\log(1-0.01) = 0.01$ tiny ✅
- $p = 0.99$ → $-\log(1-0.99) = 4.6$ huge 💀

Combine both cases into ONE formula using a mathematical switch:

$$\boxed{L = -y\log(p) - (1-y)\log(1-p)}$$

**Check the switch:**
- When $y=1$ → $(1-y)=0$ → second term vanishes → only $-\log(p)$ survives ✅
- When $y=0$ → $(1-y)=1$ → first term vanishes → only $-\log(1-p)$ survives ✅

### One Line
> **How confidently wrong you are = how hard you get punished.**

---
## 6. Cost Function

### Intuition
- **Loss** = error for ONE patient
- **Cost** = average loss over ALL patients

### Formula
$$\boxed{J(w,b) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log(p^{(i)}) + (1-y^{(i)})\log(1-p^{(i)})\right]}$$

where $p^{(i)} = \sigma(\vec{w} \cdot \vec{x}^{(i)} + b)$

### Why This Works
This cost function IS convex when combined with sigmoid. One smooth bowl. Gradient descent finds the bottom every time. ✅

---
## 7. Partial Derivatives (Gradient)

### The Chain — How w connects to J
$$w \rightarrow z \rightarrow p \rightarrow J$$

Chain rule multiplies each tiny change:
$$\frac{\partial J}{\partial w} = \frac{\partial J}{\partial p} \times \frac{\partial p}{\partial z} \times \frac{\partial z}{\partial w}$$

### Deriving Each Piece

**Piece 1:** $\frac{\partial z}{\partial w_j}$

$z = w \cdot x + b$ → differentiating w.r.t $w_j$, $x$ is the coefficient:
$$\frac{\partial z}{\partial w_j} = x_j$$

**Piece 2:** $\frac{\partial p}{\partial z}$ — Sigmoid Derivative

$p = (1+e^{-z})^{-1}$

Using chain rule:
$$\frac{\partial p}{\partial z} = -(1+e^{-z})^{-2} \times (-e^{-z}) = \frac{e^{-z}}{(1+e^{-z})^2}$$

Split the fraction:
$$= \frac{1}{1+e^{-z}} \times \frac{e^{-z}}{1+e^{-z}} = p \times (1-p)$$

$$\boxed{\frac{\partial p}{\partial z} = p(1-p)}$$

Beautiful property: uncertain model (p=0.5) changes fast. Confident model barely moves.

**Piece 3:** $\frac{\partial J}{\partial p}$

$$J = -y\log(p) - (1-y)\log(1-p)$$

$$\frac{\partial J}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p}$$

### Multiply All Three — Beautiful Cancellation
$$\frac{\partial J}{\partial w_j} = \left(-\frac{y}{p} + \frac{1-y}{1-p}\right) \times p(1-p) \times x_j$$

Expanding and simplifying (p terms cancel):

$$\boxed{\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)} - y^{(i)})x_j^{(i)}}$$

$$\boxed{\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)} - y^{(i)})}$$

**Identical shape to Linear Regression gradient!** Only difference — $p$ is now sigmoid output, not linear.

---
## 8. Regularization

### The Problem — Overfitting
Model tries SO hard to touch every training point — like a student memorizing every answer instead of understanding.

Test time → completely lost. Because it memorized, not learned.

**Mathematically — overfitting = huge weights. Always.**

To fit every point, weights become massive like $w_3 = 10000$. Huge weights = sharp bends = memorization.

### The Fix — Punish Big Weights
> *"How much mess you made, that much you clean."* — Hostel Warden Rule 😄

Add weights to cost function. Big weights → big cost → model gets punished → stays small.

Square them so negative weights also get punished (size matters, not sign):

$$\text{punishment} = \frac{\lambda}{2m}\sum_{j=1}^{n}w_j^2$$

Why $\frac{1}{2}$? So derivative cancels cleanly:
$$\frac{d}{dw_j}\left(\frac{\lambda}{2m}w_j^2\right) = \frac{\lambda}{m}w_j \text{ (no ugly 2 left over)}$$

### Regularized Cost Function
$$\boxed{J(w,b) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log(p^{(i)}) + (1-y^{(i)})\log(1-p^{(i)})\right] + \frac{\lambda}{2m}\sum_{j=1}^{n}w_j^2}$$

### What is $\lambda$?
YOUR choice. The warden's strictness dial:
```
λ = 0       → warden asleep → overfit 💀
λ = 0.01    → warden chill
λ = 1       → warden balanced ✅
λ = 10000   → warden STRICT → underfit 💀
```

### Why Not Regularize b?
$b$ just shifts the curve up or down. Doesn't cause overfitting. Warden only watches $w$. 😄

---
## 9. Gradient Descent — Final Update Rules

### For Logistic Regression (with regularization)

$$\boxed{w_j := w_j - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}(p^{(i)} - y^{(i)})x_j^{(i)} + \frac{\lambda}{m}w_j\right]}$$

$$\boxed{b := b - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}(p^{(i)} - y^{(i)})\right]}$$

where $p^{(i)} = \sigma(\vec{w} \cdot \vec{x}^{(i)} + b)$

### For Linear Regression (with regularization)

$$\boxed{w_j := w_j - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}(f^{(i)} - y^{(i)})x_j^{(i)} + \frac{\lambda}{m}w_j\right]}$$

$$\boxed{b := b - \alpha\left[\frac{1}{m}\sum_{i=1}^{m}(f^{(i)} - y^{(i)})\right]}$$

where $f^{(i)} = \vec{w} \cdot \vec{x}^{(i)} + b$

### The Only Difference
| | Linear Regression | Logistic Regression |
|--|--|--|
| $f(x)$ | $\vec{w} \cdot \vec{x} + b$ | $\sigma(\vec{w} \cdot \vec{x} + b)$ |
| Output | Any number | Probability 0→1 |
| Loss | Squared Error | Log Loss |
| Gradient shape | Same ✅ | Same ✅ |

---
## The Full Invention Chain 🔥

```
Need YES/NO probability
    ↓
Linear regression output = any number (bad)
    ↓
Threshold → kills confidence + breaks on outliers (rejected)
    ↓
Need smooth squish −∞→+∞ into 0→1
    ↓
e^x raw material + inverse trick → SIGMOID ✅
    ↓
Plug linear part into sigmoid → LOGISTIC REGRESSION ✅
    ↓
Need to train — find best w and b
    ↓
MLE — maximize probability of correct predictions
    ↓
Multiplying probabilities → underflow (rejected)
    ↓
Log turns multiply → add, preserves number
    ↓
Log of 0→1 is negative → flip sign
    ↓
NEGATIVE LOG LIKELIHOOD = LOSS FUNCTION ✅
    ↓
Average over all patients → COST FUNCTION ✅
    ↓
Chain rule on cost → GRADIENTS ✅
    ↓
Model memorizes training data → huge weights → OVERFITTING
    ↓
Punish big weights → add λ∑w² to cost
    ↓
REGULARIZATION ✅
    ↓
Final gradient descent update rules ✅
```

---
